In [ ]:
!pip install imbalanced-learn --quiet

In [39]:
import os
from tensorflow import keras
import tensorflow as tf
from keras import layers, models
import numpy as np
import kagglehub
import pandas as pd
from sklearn.utils import class_weight, shuffle
from imblearn.over_sampling import SMOTE
from keras.callbacks import EarlyStopping
import plotly.express as px
import matplotlib.pyplot as plt
import time
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
from keras.models import Sequential
from keras.layers import Input, GRU, Dense, Dropout, BatchNormalization
from sklearn.model_selection import train_test_split

In [10]:
# Download latest version
path = kagglehub.dataset_download("shayanfazeli/heartbeat")

Using Colab cache for faster access to the 'heartbeat' dataset.


In [11]:
# Salva i percorsi dei file
train_csv_path = os.path.join(path, "mitbih_train.csv")
test_csv_path = os.path.join(path, "mitbih_test.csv")

In [12]:
# Carica i csv senza intestazione
# e converte i dati da testo in numeri (float32)
df_train = pd.read_csv(train_csv_path, header=None).astype("float32")
df_test = pd.read_csv(test_csv_path, header=None).astype("float32")

In [13]:
# MAPPING DELLE CLASSI
classes = ['N', 'S', 'V', 'F', 'Q']

In [14]:
# Imposto le features e le etichette/targets
X_train = df_train.iloc[:, :-1].values # features tutte le colonne tranne l'ultima
y_train = df_train.iloc[:, -1].values # target solo l'ultima colonna

In [48]:
# DATA AUGMENTATION CON SMOTE

smote = SMOTE(random_state=42)
# 1. Split PULITO sui dati originali (senza SMOTE)
X_train_splitted, X_val_splitted, y_train_splitted, y_val_splitted = train_test_split(X_train,
                                                                                      y_train,
                                                                                      test_size=0.2, # 20% in validation
                                                                                      random_state=42, # mescola tutto
                                                                                      # assicura la presenza nelle stesse proporzioni per classe in val
                                                                                      stratify=y_train)

# 2. SMOTE SOLO sul training set!
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_splitted, y_train_splitted)

# 3. Fai il Reshape 3D per Train e Val di X (i campioni)
X_train_resampled_3D = np.expand_dims(X_train_resampled, axis=-1) # --> (87554, 187, 1)
X_val_splitted_3D = np.expand_dims(X_val_splitted, axis=-1)

# le etichette sono e rimangono 1D quindi non lo converto

In [60]:
# 1. Definizione del modello GRU
# Usiamo 64 unità GRU (esattamente come avevi fatto con la LSTM)
model_gru = Sequential([

    Input(shape=(X_train_resampled_3D.shape[1], X_train_resampled_3D.shape[2])),

    GRU(64, return_sequences=False),

    Dense(5, activation='softmax')
])

In [61]:
# Compilazione
model_gru.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy', # se y_train è one-hot, altrimenti 'sparse_categorical_crossentropy'
    metrics=['accuracy']
)

# Numero totale di parametri
model_gru.summary()

--- SUMMARY MODELLO GRU ---


Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru_9 (GRU)                     │ (None, 64)             │        12,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 13,189 (51.52 KB)

 Trainable params: 13,189 (51.52 KB)

 Non-trainable params: 0 (0.00 B)

In [53]:
# Configurazione EarlyStopping
early_stop = EarlyStopping(
    monitor='val_loss',         # Controlla la loss sul validation set
    patience=5,                 # Aspetta 4 epoche di "secca" prima di fermarsi
    restore_best_weights=True,  # memorizza i pesi MIGLIORI (non gli ultimi)
    verbose=1                   # Stampa un messaggio in console quando si attiva
)

In [54]:
# TRAINING
start_time = time.time()
history_gru = model_gru.fit(
    X_train_resampled_3D, # dati augmented 3D
    y_train_resampled, # etichette augmented 1D
    validation_data=(X_val_splitted_3D, y_val_splitted),
    epochs=30,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1
)
execution_time = time.time() - start_time
print(f"\nTempo totale di addestramento: {execution_time:.2f} secondi")

Epoch 1/30
4530/4530 ━━━━━━━━━━━━━━━━━━━━ 44s 10ms/step - accuracy: 0.7861 - loss: 0.5977 - val_accuracy: 0.7991 - val_loss: 0.6024
Epoch 2/30
4530/4530 ━━━━━━━━━━━━━━━━━━━━ 43s 9ms/step - accuracy: 0.8767 - loss: 0.3449 - val_accuracy: 0.8906 - val_loss: 0.3390
Epoch 3/30
4530/4530 ━━━━━━━━━━━━━━━━━━━━ 44s 10ms/step - accuracy: 0.9171 - loss: 0.2353 - val_accuracy: 0.9180 - val_loss: 0.2605
Epoch 4/30
4530/4530 ━━━━━━━━━━━━━━━━━━━━ 43s 9ms/step - accuracy: 0.9376 - loss: 0.1803 - val_accuracy: 0.9150 - val_loss: 0.2481
Epoch 5/30
4530/4530 ━━━━━━━━━━━━━━━━━━━━ 44s 10ms/step - accuracy: 0.9484 - loss: 0.1507 - val_accuracy: 0.9431 - val_loss: 0.1774
Epoch 6/30
4530/4530 ━━━━━━━━━━━━━━━━━━━━ 43s 9ms/step - accuracy: 0.9574 - loss: 0.1263 - val_accuracy: 0.9497 - val_loss: 0.1556
Epoch 7/30
4530/4530 ━━━━━━━━━━━━━━━━━━━━ 44s 10ms/step - accuracy: 0.9631 - loss: 0.1092 - val_accuracy: 0.9143 - val_loss: 0.2572
Epoch 8/30
4530/4530 ━━━━━━━━━━━━━━━━━━━━ 44s 10ms/step - accuracy: 0.9664 - lo

In [55]:
# PREPARAZIONE TEST

# prende solo le features perchè anche nel test file di questo dataset
# c'è la colonna labels
X_test = df_test.iloc[:, :-1].values

# converto anche il test in 3D
X_test_3D = np.expand_dims(X_test, axis=2)

In [56]:
# CLASSIFICAZIONE
predictions = model_gru.predict(X_test_3D)

685/685 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step


In [57]:
# ESTRAZIONE DATI DALLA PREDIZIONE

# calcola la probabilità e prende il massimo per riga (axis 1 cioè orizzontale)
predicted_classes = np.argmax(predictions, axis=1)

# converte le etichette numeriche in lettere comprensibili
predicted_labels = np.array(classes)[predicted_classes]

print("Classi previste per i primi 10 battiti:", predicted_labels[:10])

Classi previste per i primi 10 battiti: ['N' 'N' 'N' 'N' 'N' 'N' 'N' 'N' 'N' 'N']


In [58]:
y_test = df_test.iloc[:, -1].values
print(classification_report(y_test, predicted_classes, target_names=classes))

              precision    recall  f1-score   support

           N       0.99      0.97      0.98     18118
           S       0.58      0.81      0.67       556
           V       0.93      0.94      0.94      1448
           F       0.45      0.91      0.60       162
           Q       0.98      0.98      0.98      1608

    accuracy                           0.96     21892
   macro avg       0.79      0.92      0.83     21892
weighted avg       0.97      0.96      0.97     21892



In [59]:
# 1. Calcola le probabilità
real_values = np.argmax(y_test, axis=1) if len(y_test.shape) > 1 else y_test

# 2. Matrice di confusione in % (arrotondata a 2 decimali)
cm_percent = np.round(confusion_matrix(real_values, predicted_classes, normalize='true') * 100, 2)

# 3. Grafico Plotly
fig = px.imshow(
    cm_percent,
    x=classes,
    y=classes,
    color_continuous_scale='Blues',
    text_auto=True,  # Mostra le percentuali direttamente nelle caselle
    title="Matrice di Confusione Interattiva (%)"
)

fig.update_layout(xaxis_title="Predetto", yaxis_title="Reale")
fig.show()